# Independent Ridge-versus-COLMAP validation (CPU)

This notebook is a thin Google Colab wrapper around the repository's existing `scripts/ml/validate_independent_ridge_colmap.py`. It does not reimplement the validator, retrain a model, download CO3D, or modify preprocessing.

Before running, place these files in `MyDrive/validation/`:

- `experimental_model.joblib`
- a good-orbit video named `good_orbit.mp4` or `.mov`
- a front-only video named `front_only.mp4` or `.mov`
- an animated/waving control named `waving_control.mp4` or `.mov`

If your video stems differ, edit only `VIDEO_SPECS` below. Use a **CPU runtime**. Each run uses a new timestamped Drive directory and never overwrites earlier evidence. Three videos are a diagnostic study, not an accuracy claim.

In [ ]:
#@title 1. Configuration (normally no edits are needed)
from pathlib import Path

REPOSITORY_URL = 'https://github.com/itsmesasanda/3D-vdieo-sutability-analyzer.git'
REPOSITORY_BRANCH = 'main'
REPOSITORY_DIR = Path('/content/3D-vdieo-sutability-analyzer')
DRIVE_INPUT_DIR = Path('/content/drive/MyDrive/validation')

# video_id and role are written verbatim to the documented three-column manifest.
# filename_stem matches either MP4 or MOV, case-insensitively.
VIDEO_SPECS = [
    {'video_id': 'good_orbit', 'role': 'good_orbit', 'filename_stem': 'good_orbit'},
    {'video_id': 'front_only', 'role': 'front_only', 'filename_stem': 'front_only'},
    {'video_id': 'waving_control', 'role': 'animated_object_negative_control', 'filename_stem': 'waving_control'},
]
SUPPORTED_VIDEO_SUFFIXES = {'.mp4', '.mov'}
print('Configured repository:', REPOSITORY_URL, REPOSITORY_BRANCH)
print('Expected Drive input directory:', DRIVE_INPUT_DIR)

In [ ]:
#@title 2. Install COLMAP/Python dependencies and clone the latest committed branch
import os
import subprocess
import sys

def run_checked(command, *, cwd=None):
    print('+', ' '.join(str(item) for item in command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True, text=True)

run_checked(['apt-get', 'update', '-qq'])
run_checked([
    'apt-get', 'install', '-y', '-qq',
    'colmap', 'ffmpeg', 'libgl1', 'libglib2.0-0'
])

if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / '.git').is_dir():
        raise RuntimeError(f'{REPOSITORY_DIR} exists but is not a Git clone; use a fresh runtime')
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPOSITORY_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError('Existing Colab clone has local changes; use a fresh runtime rather than overwriting them')
    run_checked(['git', 'pull', '--ff-only', 'origin', REPOSITORY_BRANCH], cwd=REPOSITORY_DIR)
else:
    run_checked([
        'git', 'clone', '--depth', '1', '--branch', REPOSITORY_BRANCH,
        REPOSITORY_URL, str(REPOSITORY_DIR)
    ])

run_checked([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[app-ml]'], cwd=REPOSITORY_DIR)
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR, text=True).strip()
colmap_help = subprocess.check_output(['colmap', '-h'], stderr=subprocess.STDOUT, text=True)
colmap_version_line = next((line.strip() for line in colmap_help.splitlines() if line.strip()), 'unknown')

import cv2
import joblib
import numpy as np
import sklearn
from video_suitability.ml.historical_registration_features import LIVE_VIDEO_ADAPTER_CONTRACT_SHA256
from video_suitability.ml.independent_validation import FROZEN_COLMAP_SETTINGS
if FROZEN_COLMAP_SETTINGS['use_gpu'] is not False:
    raise RuntimeError('The repository validation policy is no longer CPU-only')

print('Repository commit:', commit)
print('COLMAP:', colmap_version_line)
print('Python:', sys.version.split()[0])
print('OpenCV:', cv2.__version__, '| NumPy:', np.__version__, '| scikit-learn:', sklearn.__version__)
print('Frozen adapter contract:', LIVE_VIDEO_ADAPTER_CONTRACT_SHA256)
print('Frozen COLMAP device policy: CPU (use_gpu=False)')

In [ ]:
#@title 3. Mount the Google Drive that owns the validation inputs
from google.colab import drive
drive.mount('/content/drive')

if not DRIVE_INPUT_DIR.is_dir():
    raise FileNotFoundError(
        f'Missing {DRIVE_INPUT_DIR}. Create MyDrive/validation and add the model and videos.'
    )
print('Drive inputs:')
for path in sorted(DRIVE_INPUT_DIR.iterdir()):
    if path.is_file():
        print(f'  {path.name} ({path.stat().st_size / (1024 * 1024):.2f} MiB)')

In [ ]:
#@title 4. Verify the trusted model, resolve videos, and create the documented manifest
import csv
import hashlib
import json
from datetime import datetime, timezone

from video_suitability.ml.experimental_ridge import (
    MODEL_SHA256,
    load_experimental_ridge,
)
from video_suitability.ml.independent_validation import load_validation_manifest

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def resolve_video(stem):
    matches = [
        path for path in DRIVE_INPUT_DIR.iterdir()
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_VIDEO_SUFFIXES
        and path.stem.casefold() == stem.casefold()
    ]
    if len(matches) != 1:
        available = sorted(
            path.name for path in DRIVE_INPUT_DIR.iterdir()
            if path.is_file() and path.suffix.lower() in SUPPORTED_VIDEO_SUFFIXES
        )
        raise FileNotFoundError(
            f'Expected exactly one MP4/MOV with stem {stem!r}; found {len(matches)}. '
            f'Available videos: {available}. Edit VIDEO_SPECS if needed.'
        )
    return matches[0].resolve()

MODEL_PATH = (DRIVE_INPUT_DIR / 'experimental_model.joblib').resolve()
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Missing trusted model artifact: {MODEL_PATH}')
actual_model_sha256 = sha256_file(MODEL_PATH)
if actual_model_sha256 != MODEL_SHA256:
    raise RuntimeError(
        f'Model integrity check failed: expected {MODEL_SHA256}, got {actual_model_sha256}'
    )

ARTIFACT_DIRECTORY = (
    REPOSITORY_DIR / 'evaluation/ml/historical_registration_experiment/historical_opencv8_v2'
).resolve()
_, model_integrity = load_experimental_ridge(
    artifact_directory=ARTIFACT_DIRECTORY,
    model_bytes=MODEL_PATH.read_bytes(),
)
print('Model integrity/parity:', json.dumps(model_integrity, indent=2))

resolved = []
for specification in VIDEO_SPECS:
    video_path = resolve_video(specification['filename_stem'])
    resolved.append({**specification, 'video_path': video_path})

RUN_ID = 'ridge_colmap_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_RUN_ROOT = DRIVE_INPUT_DIR / 'results' / RUN_ID
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=False)
MANIFEST_PATH = DRIVE_RUN_ROOT / 'validation_videos.csv'
with MANIFEST_PATH.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['video_id', 'role', 'video_path'])
    writer.writeheader()
    for item in resolved:
        writer.writerow({
            'video_id': item['video_id'],
            'role': item['role'],
            'video_path': str(item['video_path']),
        })

# Exercise the repository's own manifest parser before starting expensive work.
validated_records = load_validation_manifest(MANIFEST_PATH)
inventory = {
    'repository_url': REPOSITORY_URL,
    'repository_branch': REPOSITORY_BRANCH,
    'repository_commit': commit,
    'model_path': str(MODEL_PATH),
    'model_sha256': actual_model_sha256,
    'adapter_contract_sha256': LIVE_VIDEO_ADAPTER_CONTRACT_SHA256,
    'videos': [
        {
            'video_id': item['video_id'],
            'role': item['role'],
            'path': str(item['video_path']),
            'size_bytes': item['video_path'].stat().st_size,
            'sha256': sha256_file(item['video_path']),
        }
        for item in resolved
    ],
}
(DRIVE_RUN_ROOT / 'input_inventory.json').write_text(
    json.dumps(inventory, indent=2) + '\n', encoding='utf-8'
)
print(MANIFEST_PATH.read_text(encoding='utf-8'))
print('Validated records:', len(validated_records))
print('Persistent run directory:', DRIVE_RUN_ROOT)

In [ ]:
#@title 5. Run the unchanged frozen adapter + Ridge + CPU COLMAP validator
import shutil
import time

LOCAL_OUTPUT = Path('/content') / RUN_ID
DRIVE_EVIDENCE = DRIVE_RUN_ROOT / 'evidence'
DRIVE_EVIDENCE.mkdir(parents=True, exist_ok=False)
if LOCAL_OUTPUT.exists():
    raise RuntimeError(f'Local output already exists: {LOCAL_OUTPUT}; rerun the manifest cell for a new RUN_ID')

# Only concise evidence is synchronized periodically. Prepared JPEGs, databases,
# sparse models and point clouds stay in Colab scratch storage.
TOP_LEVEL_EVIDENCE = {
    'validation_results.csv', 'VALIDATION_REPORT.md',
    'validation_summary.json', 'validation_config.json',
}
RUN_EVIDENCE = {
    'validation_record.json', 'preparation_manifest.json',
    'ridge_prediction.json', 'reconstruction_result.json',
}

def sync_evidence():
    copied = 0
    if not LOCAL_OUTPUT.exists():
        return copied
    for source in LOCAL_OUTPUT.rglob('*'):
        if not source.is_file():
            continue
        relative = source.relative_to(LOCAL_OUTPUT)
        keep = (
            (len(relative.parts) == 1 and source.name in TOP_LEVEL_EVIDENCE)
            or source.name in RUN_EVIDENCE
            or source.suffix.lower() == '.log'
        )
        if not keep:
            continue
        destination = DRIVE_EVIDENCE / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        copied += 1
    return copied

command = [
    sys.executable,
    'scripts/ml/validate_independent_ridge_colmap.py',
    '--manifest', str(MANIFEST_PATH),
    '--output-dir', str(LOCAL_OUTPUT),
    '--model-artifact', str(MODEL_PATH),
    '--artifact-directory', str(ARTIFACT_DIRECTORY),
    '--colmap', 'colmap',
]
(DRIVE_RUN_ROOT / 'validator_command.json').write_text(
    json.dumps({'command': command, 'cwd': str(REPOSITORY_DIR)}, indent=2) + '\n',
    encoding='utf-8',
)
CONSOLE_LOG = DRIVE_RUN_ROOT / 'validator_console.log'
print('+', ' '.join(command), flush=True)
started = time.monotonic()
with CONSOLE_LOG.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        command, cwd=REPOSITORY_DIR, stdout=log_handle,
        stderr=subprocess.STDOUT, text=True, env=os.environ.copy(),
    )
    while process.poll() is None:
        copied = sync_evidence()
        print(
            f'Validator active: {time.monotonic() - started:.0f}s elapsed; '
            f'{copied} evidence files synchronized to Drive',
            flush=True,
        )
        time.sleep(15)
    return_code = process.wait()
copied = sync_evidence()
print(f'Validator exit code: {return_code}; final synchronized files: {copied}')

console_text = CONSOLE_LOG.read_text(encoding='utf-8', errors='replace')
print(console_text[-6000:] if console_text else '(validator produced no console output)')
if return_code == 3:
    print('WARNING: no valid completed COLMAP ground truth exists; MAE was intentionally not calculated.')
elif return_code != 0:
    raise RuntimeError(
        f'Validation setup failed with exit code {return_code}. See {CONSOLE_LOG}'
    )

In [ ]:
#@title 6. Display predictions, verified COLMAP results, status, runtime, and valid-only MAE
import pandas as pd
from IPython.display import display, Markdown

RESULTS_CSV = LOCAL_OUTPUT / 'validation_results.csv'
if not RESULTS_CSV.is_file():
    raise FileNotFoundError(
        f'No result CSV was produced. Inspect {CONSOLE_LOG} rather than assuming any video was skipped.'
    )
results = pd.read_csv(RESULTS_CSV)
valid_ground_truth = (
    results['ground_truth_valid'].astype(str).str.lower().eq('true')
    & results['actual_registered_images'].notna()
)
valid_errors = valid_ground_truth & results['absolute_error_percentage_points'].notna()

device_by_video = {}
for video_id in results['video_id'].astype(str):
    reconstruction_path = LOCAL_OUTPUT / 'runs' / video_id / 'colmap' / 'reconstruction_result.json'
    if reconstruction_path.is_file():
        reconstruction = json.loads(reconstruction_path.read_text(encoding='utf-8'))
        device_by_video[video_id] = reconstruction.get('selected_device', 'unresolved')
    else:
        device_by_video[video_id] = 'not reached'

table = results[[
    'video_id', 'role', 'predicted_registration_percent',
    'actual_registered_images', 'total_input_images',
    'actual_registration_percent', 'absolute_error_percentage_points',
    'reconstruction_status', 'total_execution_seconds',
]].copy()
table.columns = [
    'Video', 'Role', 'Predicted registration (%)',
    'Actual registered frames', 'Input frames',
    'Actual registration (%)', 'Absolute error (pp)',
    'Execution status', 'Runtime (s)',
]
table.insert(8, 'COLMAP device', table['Video'].astype(str).map(device_by_video))
completed_devices = set(table.loc[valid_ground_truth, 'COLMAP device'])
if completed_devices and completed_devices != {'cpu'}:
    raise RuntimeError(f'Valid COLMAP results were not CPU-only: {completed_devices}')
display(table)

summary = json.loads((LOCAL_OUTPUT / 'validation_summary.json').read_text(encoding='utf-8'))
if valid_errors.any():
    checked_mae = float(results.loc[valid_errors, 'absolute_error_percentage_points'].mean())
    if abs(checked_mae - float(summary['mae_percentage_points'])) > 1e-6:
        raise RuntimeError('Displayed valid-only MAE differs from the validator summary')
    display(Markdown(
        f"**Diagnostic MAE:** {checked_mae:.2f} percentage points over "
        f"{int(valid_errors.sum())} valid completed prediction/ground-truth pairs."
    ))
else:
    display(Markdown(
        '**Diagnostic MAE: Not calculated.** No valid completed COLMAP result '        'with an available Ridge prediction exists.'
    ))

failed = results.loc[~valid_ground_truth, [
    'video_id', 'reconstruction_status', 'raw_colmap_status',
    'failure_stage', 'failure_reason',
]]
if not failed.empty:
    display(Markdown('### Invalid or incomplete ground-truth evaluations (excluded from MAE)'))
    display(failed)

print(json.dumps(summary, indent=2))
print('CSV:', DRIVE_EVIDENCE / 'validation_results.csv')
print('Report:', DRIVE_EVIDENCE / 'VALIDATION_REPORT.md')
print('Console log:', CONSOLE_LOG)

In [ ]:
#@title 7. Package the lightweight persistent evidence
# The evidence directory intentionally excludes prepared images, COLMAP databases,
# sparse models and point clouds. Those remain in /content until the runtime ends.
sync_evidence()
archive_base = DRIVE_RUN_ROOT / 'ridge_colmap_validation_evidence'
archive_path = Path(shutil.make_archive(
    str(archive_base), 'zip', root_dir=DRIVE_RUN_ROOT,
    base_dir='evidence',
))
print('Persistent Drive directory:', DRIVE_RUN_ROOT)
print('Evidence ZIP:', archive_path)
print('Keep validation_videos.csv, input_inventory.json, validator_command.json,')
print('validator_console.log, the evidence folder, and this ZIP together.')
print('Do not interpret three videos as a statistically sufficient accuracy evaluation.')